In [1]:
import geopandas as gpd
import pandas as pd
from shapely import wkb

In [2]:
ph_bounds = gpd.read_file(" ../../data/ph_adm3_municities/PH_Adm3_MuniCities.shp.shp") 
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
0,100000000,102800000,102801000,Adams,Mun,45997,111184551,45,111.0,"POLYGON ((285598.94 2047831, 284434.314 204265..."
1,100000000,102800000,102802000,Bacarra,Mun,33313,55346073,33,55.0,"POLYGON ((253498.433 2023518.973, 253089.772 2..."
2,100000000,102800000,102803000,Badoc,Mun,64985,80758428,64,80.0,"POLYGON ((232924.209 1989474.502, 232926.91 19..."
3,100000000,102800000,102804000,Bangui,Mun,52068,115127442,52,115.0,"POLYGON ((269159.821 2050731.232, 269201.186 2..."
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2..."


In [3]:
groundsource_df = pd.read_parquet("../../data/groundsource.parquet")
groundsource_df.head()

,uuid,area_km2,geometry,start_date,end_date
0,5acc1866dd6644dfa572f02ae3d54aa4,91.678015,"b""\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...",2000-01-01,2000-01-01
1,f80fccbefb1346a9b568907086e65226,609.968365,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...,2000-01-01,2000-01-01
2,d5c49c9e2c2045bfbfd6e3b3865efc3f,0.099688,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2000-01-24,2000-01-24
3,ce7f01a29de040b39e138f198fc39d86,0.548906,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2000-01-30,2000-01-30
4,eb047dfea9ed406485010853226e0170,0.236562,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2000-01-30,2000-01-30


In [4]:
# Use wkb.loads to turn bytes into a Shapely object
groundsource_df['geometry'] = groundsource_df['geometry'].apply(lambda x: wkb.loads(x))

# Create a GeoDataFrame with CRS being EPSG:4326 (WGS 84) as per the Groundsource docs
gs_gdf = gpd.GeoDataFrame(groundsource_df, geometry='geometry', crs="EPSG:4326")

# Create the joining point for the spatial join
gs_gdf['join_point'] = gs_gdf.geometry.representative_point()

In [20]:
# Standardize both datasets to a common CRS
gs_gdf = gs_gdf.to_crs(epsg=4326)
ph_admin = ph_bounds.to_crs(epsg=4326)

# Use the Representative Point for the Groundsource data, which
# turns complex shapes into a single 'seed' point inside the area
gs_gdf['join_geometry'] = gs_gdf.geometry.representative_point()

# Use a Spatial Join (sjoin) with predicate within, basically asking if 
# the representative point is inside the city/municipality's border
tagged_df = gpd.sjoin(
    gs_gdf.set_geometry('join_geometry'), 
    ph_admin[['adm3_en', 'geo_level', 'geometry']],
    how="left", 
    predicate="within" 
)

In [36]:
# Getting only records inside the PH Cities/Municipalities
cleaned_gdf = tagged_df[~tagged_df.adm3_en.isna()]
cleaned_gdf.head()

,uuid,area_km2,geometry,start_date,end_date,join_point,join_geometry,index_right,adm3_en,geo_level
677,22d1378d5800436a82b25963b9b3ef86,0.021353,"POLYGON ((120.98918 14.5835, 120.97716 14.5789...",2001-02-03,2001-02-03,POINT (120.98304 14.5812),POINT (120.98304 14.5812),1280.0,City of Manila,City
1020,2338c789bd9d4b9095d51cdd52bf70b8,286.858664,"MULTIPOLYGON (((123.54344 10.96718, 123.55643 ...",2001-11-04,2001-11-04,POINT (123.32426 10.74773),POINT (123.32426 10.74773),720.0,City of Sagay,City
1021,b7dbf5cdee8c451c96c25e39e8e7f44e,89.129582,"POLYGON ((123.93304 10.84286, 123.96106 10.890...",2001-11-04,2001-11-04,POINT (123.99411 10.86314),POINT (123.99411 10.86314),797.0,Borbon,Mun
3505,0ebe76e3080e495a98e02c2b78fc3eab,1673.357384,"MULTIPOLYGON (((125.11603 10.14519, 125.1287 1...",2003-12-15,2003-12-21,POINT (125.15072 10.33319),POINT (125.15072 10.33319),986.0,Hinunangan,Mun
3506,d0ae67c8eafb478c86950c32835e8adb,287.471679,"MULTIPOLYGON (((125.69612 9.79255, 125.722 9.8...",2003-12-19,2003-12-23,POINT (125.49828 9.74559),POINT (125.49828 9.74559),1412.0,City of Surigao,City


In [35]:
# Filtering to only get city flood events from 2022 to 2025
urban_flood_gdf = cleaned_gdf[
    ((cleaned_gdf["geo_level"] == "City") &
    (cleaned_gdf["start_date"] >= "2022-01-01")) &
    (cleaned_gdf["start_date"] < "2026-01-01")
]
urban_flood_gdf.head()

,uuid,area_km2,geometry,start_date,end_date,join_point,join_geometry,index_right,adm3_en,geo_level
1383223,eab0bfe86aeb45cdbd1859e2aae325f9,212.500434,"POLYGON ((122.94193 10.85978, 122.98532 10.832...",2022-01-01,2022-01-02,POINT (123.09825 10.76178),POINT (123.09825 10.76178),723.0,City of Silay,City
1383347,85b0ede56bd84e9ab702a62a0ffad913,287.471679,"MULTIPOLYGON (((125.69612 9.79255, 125.722 9.8...",2022-01-01,2022-01-02,POINT (125.49828 9.74559),POINT (125.49828 9.74559),1412.0,City of Surigao,City
1387873,f3d20f47ab3c474aa7a57879a9a2afe9,42.820100,"POLYGON ((120.9328 14.6029, 120.95421 14.6011,...",2022-01-09,2022-01-09,POINT (120.97974 14.59863),POINT (120.97974 14.59863),1280.0,City of Manila,City
1392368,c83422337aac454f824b2db8f464231d,11.513759,"POLYGON ((125.74019 7.36969, 125.75602 7.40073...",2022-01-18,2022-01-23,POINT (125.75989 7.38559),POINT (125.75989 7.38559),1184.0,City of Tagum,City
1392480,18e957ca690b44a79d5d24c64b993bb1,180.059998,"POLYGON ((125.73025 7.33202, 125.76174 7.41692...",2022-01-18,2022-01-18,POINT (125.79381 7.39147),POINT (125.79381 7.39147),1184.0,City of Tagum,City


In [34]:
# Deduplication and removing unnecessary columns
urban_flood_df = urban_flood_gdf[[
    "adm3_en",
    "geo_level",
    "start_date",
    "end_date"
]].drop_duplicates().reset_index(drop=True)

urban_flood_df

,adm3_en,geo_level,start_date,end_date
0,City of Silay,City,2022-01-01,2022-01-02
1,City of Surigao,City,2022-01-01,2022-01-02
2,City of Manila,City,2022-01-09,2022-01-09
3,City of Tagum,City,2022-01-18,2022-01-23
4,City of Tagum,City,2022-01-18,2022-01-18
...,...,...,...,...
3186,City of San Pablo,City,2025-12-09,2025-12-11
3187,City of Manila,City,2025-12-10,2025-12-10
3188,City of Iloilo,City,2025-12-11,2025-12-11
3189,City of Cebu,City,2025-12-23,2025-12-24


In [37]:
urban_flood_df.to_csv("../../data/urban_flood_groundsource_df.csv", index=False)